# Lung Cancer Detection & Localization — Two-Stage Pipeline

This notebook documents a two-stage deep learning pipeline for detecting and
localizing lung tumors (nodules) in imaging data, and classifying them as
**benign** or **malignant**.

## Motivation

An object detector alone tends to produce **false positives** — bounding
boxes around regions that look tumor-like but aren't. To fix this, a
second-stage classifier is used to re-check every box the detector proposes
and discard the ones that don't hold up.

## Pipeline overview

| Stage | Model | Role |
|---|---|---|
| 1. Detection | Faster R-CNN with a ResNet-152 + FPN backbone | Proposes candidate bounding boxes for suspicious regions |
| 2. Classification | ResNet-50 (binary) | Re-classifies each cropped box as a true finding or a false positive, and assigns benign/malignant |

An **adaptive confidence threshold** is used at inference time: detection and
classification thresholds start high (0.9) and are gradually relaxed in steps
of 0.05 down to a floor of 0.05 until at least one confident detection
survives. This keeps precision high on easy images while still recovering
faint findings on harder ones.

## Final validation results

| Metric | Value |
|---|---|
| Precision | **0.8734** |
| Recall | **0.9979** |
| F1-Score | **0.9315** |
| Mean IoU | **0.8104** |

The high recall (~0.998) confirms the detector rarely misses a true nodule,
while the two-step classification filter raises precision to ~0.87 by
rejecting most of the detector's false positives — this was the specific
issue this pipeline was built to resolve.


## 1. Setup & Imports

In [ ]:
import os
import xml.etree.ElementTree as ET

import torch
from torch.nn import BCEWithLogitsLoss
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, random_split

import xmltodict
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from tqdm import tqdm

import torchvision.transforms as T
from torchvision.ops import nms, box_iou
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 2. Configuration

Central place for paths and thresholds used throughout the notebook. Update
the `*_DIR` and `*_CKPT` paths to match your environment before running.

In [ ]:
# --- Data paths (PASCAL-VOC style: JPEGImages/ + Annotations/) ---
TRAIN_IMG_DIR = "dataset/JPEGImages"
TRAIN_ANN_DIR = "dataset/Annotations"
VAL_IMG_DIR = "dataset/val/JPEGImages"
VAL_ANN_DIR = "dataset/val/Annotations"

# --- Checkpoints ---
DETECTOR_CKPT = "detector.ckpt"
CLASSIFIER_CKPT = "classifier.ckpt"

# --- Classifier (stage 2) classes ---
CLASS_NAMES = {0: "benign", 1: "malignant"}

# --- Adaptive thresholding for inference ---
INITIAL_DET_THRESH = 0.9   # detector confidence to start with
INITIAL_CLS_THRESH = 0.9   # classifier confidence to start with
MIN_THRESH = 0.05          # floor both thresholds can relax down to
STEP = 0.05                # amount to relax thresholds by each iteration
NMS_IOU_THRESHOLD = 0.3    # IoU threshold for non-max suppression
IOU_THRESHOLD = 0.5        # IoU threshold to count a prediction as a match during evaluation


## 3. Dataset

`VOCDataset` reads PASCAL-VOC style annotations (`Annotations/*.xml`) paired
with images (`JPEGImages/*.png`) and returns `(image_tensor, target_dict)`
pairs suitable for `torchvision`'s detection models. Boxes are labeled `1`
for a cancer/nodule finding and `2` for any other annotated object.

In [ ]:
class VOCDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir, self.ann_dir = img_dir, ann_dir
        self.ids = [f.split(".")[0] for f in os.listdir(img_dir)]
        self.transforms = transforms

    def __getitem__(self, i):
        img_id = self.ids[i]
        img = Image.open(os.path.join(self.img_dir, img_id + ".png")).convert("RGB")
        doc = xmltodict.parse(open(os.path.join(self.ann_dir, img_id + ".xml")).read())
        objs = doc["annotation"].get("object", [])
        if isinstance(objs, dict):
            objs = [objs]  # handle the single-object case

        boxes, labels = [], []
        for obj in objs:
            name = obj["name"]
            coords = obj["bndbox"]
            boxes.append([float(coords["xmin"]), float(coords["ymin"]),
                          float(coords["xmax"]), float(coords["ymax"])])
            labels.append(1 if name == "cancer" else 2)  # 1 = cancer, 2 = other/background

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([i]), "original_id": img_id}

        img_tensor = self.transforms(img) if self.transforms else T.ToTensor()(img)
        return img_tensor, target

    def __len__(self):
        return len(self.ids)


def collate_fn(batch):
    return tuple(zip(*batch))


## 4. Stage 1 — Detector (Faster R-CNN, ResNet-152 + FPN)

Wraps `torchvision`'s `FasterRCNN` with a ResNet-152 FPN backbone in a
`pytorch_lightning.LightningModule`. Custom anchor sizes/aspect ratios are
defined to better match the scale of lung nodules across the 5 FPN levels.
`num_classes=3` accounts for background + the two annotated classes above.

In [ ]:
class Detector(pl.LightningModule):
    def __init__(self, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()

        backbone = resnet_fpn_backbone("resnet152", pretrained=True)

        sizes = ((32,), (64,), (128,), (256,), (512,))
        aspect_ratios = ((0.5, 1.0, 2.0),) * len(sizes)
        anchor_generator = AnchorGenerator(sizes=list(sizes), aspect_ratios=list(aspect_ratios))

        self.model = FasterRCNN(backbone, num_classes=3, rpn_anchor_generator=anchor_generator)
        self.lr = lr

    def forward(self, images):
        return self.model(images)

    def training_step(self, batch, idx):
        images, targets = batch
        loss_dict = self.model(images, targets)
        loss = sum(loss_dict.values())
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, idx):
        images, targets = batch
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(images)
        return outputs

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


## 5. Stage 2 — Classifier (ResNet-50, binary)

A pretrained ResNet-50 with its final FC layer replaced for single-logit
binary output. This model re-inspects each crop the detector proposes and
decides whether it's a genuine finding (and benign vs. malignant), which is
what drives down the detector's false-positive rate.

In [ ]:
class TumorClassifier(pl.LightningModule):
    def __init__(self, num_classes=1, learning_rate=1e-4):
        super().__init__()
        self.save_hyperparameters()

        weights = ResNet50_Weights.DEFAULT
        self.backbone = resnet50(weights=weights)
        num_ftrs = self.backbone.fc.in_features
        self.backbone.fc = torch.nn.Linear(num_ftrs, num_classes)

        self.loss_fn = BCEWithLogitsLoss()

    def forward(self, x):
        return self.backbone(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.loss_fn(logits, labels)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.loss_fn(logits, labels)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True, logger=True)

        preds = torch.sigmoid(logits) > 0.5
        accuracy = (preds == labels).float().mean()
        self.log("val_accuracy", accuracy, on_step=False, on_epoch=True, prog_bar=True, logger=True)

    def test_step(self, batch, batch_idx):
        images, labels = batch
        logits = self(images)
        loss = self.loss_fn(logits, labels)
        self.log("test_loss", loss)

        preds = torch.sigmoid(logits) > 0.5
        accuracy = (preds == labels).float().mean()
        self.log("test_accuracy", accuracy)

    def configure_optimizers(self):
        return Adam(self.parameters(), lr=self.hparams.learning_rate)


## 6. Training

Both stages are trained independently with `pytorch_lightning.Trainer`.
The detector learns to localize candidate regions; the classifier is
trained separately on cropped patches (true findings vs. false-positive
crops sampled from the detector, plus benign/malignant labels) so it can
later veto bad detector proposals.

> These cells assume `train_dataset` / `val_dataset` (detector) and a
> `TumorCropDataset`-style loader (classifier) are available in your data
> directories. Adjust `DataLoader` args (batch size, workers) to fit your
> hardware.

In [ ]:
# --- Stage 1: train the detector ---
detector_transform = T.ToTensor()

train_dataset = VOCDataset(TRAIN_IMG_DIR, TRAIN_ANN_DIR, transforms=detector_transform)
train_len = int(0.9 * len(train_dataset))
val_len = len(train_dataset) - train_len
det_train_ds, det_val_ds = random_split(train_dataset, [train_len, val_len])

det_train_loader = DataLoader(det_train_ds, batch_size=4, shuffle=True,
                               num_workers=4, collate_fn=collate_fn)
det_val_loader = DataLoader(det_val_ds, batch_size=4, shuffle=False,
                             num_workers=4, collate_fn=collate_fn)

detector = Detector(lr=1e-4)

detector_checkpoint = ModelCheckpoint(
    dirpath=".", filename="detector", monitor="train_loss", mode="min", save_top_k=1
)

detector_trainer = pl.Trainer(
    max_epochs=30,
    accelerator="auto",
    callbacks=[detector_checkpoint],
)

# detector_trainer.fit(detector, det_train_loader, det_val_loader)


In [ ]:
# --- Stage 2: train the classifier ---
classifier_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Expected layout: CLASSIFIER_DATA_DIR/{benign,malignant}/*.png
# CLASSIFIER_DATA_DIR = "dataset/classifier_crops"
# full_ds = TumorCropDataset(CLASSIFIER_DATA_DIR, transforms=classifier_transform)
# train_len = int(0.8 * len(full_ds))
# cls_train_ds, cls_val_ds = random_split(full_ds, [train_len, len(full_ds) - train_len])

# cls_train_loader = DataLoader(cls_train_ds, batch_size=32, shuffle=True, num_workers=4)
# cls_val_loader = DataLoader(cls_val_ds, batch_size=32, shuffle=False, num_workers=4)

classifier = TumorClassifier(learning_rate=1e-4)

classifier_checkpoint = ModelCheckpoint(
    dirpath=".", filename="classifier", monitor="val_loss", mode="min", save_top_k=1
)
early_stop = EarlyStopping(monitor="val_loss", patience=5, mode="min")

classifier_trainer = pl.Trainer(
    max_epochs=25,
    accelerator="auto",
    callbacks=[classifier_checkpoint, early_stop],
)

# classifier_trainer.fit(classifier, cls_train_loader, cls_val_loader)


## 7. Two-Stage Inference Pipeline

`detect_and_classify` runs both models on a single image:

1. Run the detector, keep boxes above `det_thresh`, and apply NMS.
2. Crop each surviving box and pass it through the classifier.
3. Keep only crops the classifier is confident about (above `cls_thresh`)
   — this is the step that removes the detector's false positives.
4. If nothing survives, relax both thresholds by `STEP` and retry, down to
   `MIN_THRESH`, so a genuine (but faint) finding still gets reported.

`draw_detections` then overlays the surviving boxes and labels for visual
inspection.

In [ ]:
def _safe_crop_box(box, w, h):
    '''Clamp a box to valid image bounds; return None if it collapses.'''
    x1, y1, x2, y2 = map(int, box)
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(0, min(x2, w))
    y2 = max(0, min(y2, h))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def detect_and_classify(image: Image.Image, detector, classifier):
    '''Run the two-stage pipeline on a single PIL image.

    Returns (boxes, labels, probs) for every detection that survives both
    the detector's and classifier's confidence thresholds.
    '''
    img_w, img_h = image.size
    img_tensor = detector_transform(image).unsqueeze(0).to(DEVICE)

    det_thresh, cls_thresh = INITIAL_DET_THRESH, INITIAL_CLS_THRESH
    final_boxes, final_labels, final_probs = [], [], []

    while det_thresh >= MIN_THRESH:
        with torch.no_grad():
            outputs = detector(img_tensor)[0]

        boxes = outputs.get("boxes", torch.empty((0, 4), device=DEVICE))
        scores = outputs.get("scores", torch.empty((0,), device=DEVICE))

        keep = scores > det_thresh
        boxes, scores = boxes[keep], scores[keep]

        if boxes.numel() > 0:
            keep_nms = nms(boxes, scores, iou_threshold=NMS_IOU_THRESHOLD)
            boxes, scores = boxes[keep_nms], scores[keep_nms]

        for box in boxes:
            box = _safe_crop_box(box.cpu().tolist(), img_w, img_h)
            if box is None:
                continue
            x1, y1, x2, y2 = box
            crop = image.crop((x1, y1, x2, y2))
            if crop.width < 1 or crop.height < 1:
                continue

            with torch.no_grad():
                crop_tensor = classifier_transform(crop).unsqueeze(0).to(DEVICE)
                logits = classifier(crop_tensor)
                prob = torch.sigmoid(logits).flatten()[0].item()
                pred_class = int(prob > 0.5)

            if prob > cls_thresh:
                final_boxes.append([x1, y1, x2, y2])
                final_labels.append(CLASS_NAMES[pred_class])
                final_probs.append(prob)

        if final_boxes:
            break
        det_thresh -= STEP
        cls_thresh -= STEP

    return final_boxes, final_labels, final_probs


def _get_font(size=16):
    '''Best-effort font loader across platforms.'''
    try:
        return ImageFont.truetype("arial.ttf", size)
    except Exception:
        try:
            return ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size)
        except Exception:
            return ImageFont.load_default()


def draw_detections(image: Image.Image, boxes, labels, probs=None):
    '''Return a copy of `image` annotated with the given boxes/labels.'''
    img = image.copy()
    draw = ImageDraw.Draw(img)
    font = _get_font(16)

    for i, (box, label) in enumerate(zip(boxes, labels)):
        x1, y1, x2, y2 = box
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        text = label + (f" ({probs[i]:.2f})" if probs is not None else "")
        tw, th = draw.textbbox((0, 0), text, font=font)[2:]
        draw.rectangle([x1, y1 - th - 4, x1 + tw + 4, y1], fill="red")
        draw.text((x1 + 2, y1 - th - 2), text, fill="white", font=font)

    return img


## 8. Load Trained Models

Load both checkpoints for evaluation and inference below.

In [ ]:
detector = Detector.load_from_checkpoint(DETECTOR_CKPT).to(DEVICE).eval()
classifier = TumorClassifier.load_from_checkpoint(CLASSIFIER_CKPT).to(DEVICE).eval()
print("Loaded detector and classifier checkpoints.")


## 9. Single-Image Inference Example

Run the pipeline on one image and visualize the surviving detections.

In [ ]:
target_image_path = "sample.png"  # <- point this at any test image

image = Image.open(target_image_path).convert("RGB")
boxes, labels, probs = detect_and_classify(image, detector, classifier)

if not boxes:
    print("No tumors detected.")
else:
    print("Detected tumors:")
    for box, label, p in zip(boxes, labels, probs):
        print(f" - {label:9s} {p:.2f} at {box}")

annotated = draw_detections(image, boxes, labels, probs if boxes else None)

plt.figure(figsize=(8, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(os.path.basename(target_image_path))
plt.show()


## 10. Full Validation & Metrics

Runs the two-stage pipeline over an entire validation set with
ground-truth VOC annotations and computes precision, recall, F1, and mean
IoU. A prediction counts as a true positive if its IoU with some
ground-truth box is at least `IOU_THRESHOLD`.

In [ ]:
def parse_voc_annotation(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        labels.append(obj.find("name").text)
        b = obj.find("bndbox")
        boxes.append([int(b.find("xmin").text), int(b.find("ymin").text),
                      int(b.find("xmax").text), int(b.find("ymax").text)])
    return torch.tensor(boxes, dtype=torch.float32), labels


true_positives = 0
false_positives = 0
false_negatives = 0
ious_list = []

image_files = [f for f in os.listdir(VAL_IMG_DIR) if f.endswith(".png")]

for img_file in tqdm(image_files, desc="Validating"):
    img_path = os.path.join(VAL_IMG_DIR, img_file)
    ann_path = os.path.join(VAL_ANN_DIR, img_file.replace(".png", ".xml"))

    img = Image.open(img_path).convert("RGB")
    gt_boxes, gt_labels = parse_voc_annotation(ann_path)

    final_boxes, final_labels, _ = detect_and_classify(img, detector, classifier)
    final_boxes = [torch.tensor(b, dtype=torch.float32) for b in final_boxes]

    if final_boxes and len(gt_boxes) > 0:
        ious = box_iou(torch.stack(final_boxes), gt_boxes)
        max_ious, _ = ious.max(dim=1)
        for iou in max_ious:
            if iou >= IOU_THRESHOLD:
                true_positives += 1
                ious_list.append(iou.item())
            else:
                false_positives += 1
        false_negatives += max(0, len(gt_boxes) - true_positives)
    else:
        if final_boxes:
            false_positives += len(final_boxes)
        if len(gt_boxes) > 0:
            false_negatives += len(gt_boxes)

precision = true_positives / (true_positives + false_positives + 1e-6)
recall = true_positives / (true_positives + false_negatives + 1e-6)
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
mean_iou = sum(ious_list) / len(ious_list) if ious_list else 0.0

print("\n--- Validation Results ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Mean IoU:  {mean_iou:.4f}")


### Reported results

Running the cell above on the full validation set produced the final
numbers for this pipeline:

```
Precision: 0.8734
Recall:    0.9979
F1-Score:  0.9315
Mean IoU:  0.8104
```

**Takeaway:** recall near 1.0 shows the detector almost never misses a real
nodule; the classifier stage is what lifts precision from a lower,
false-positive-heavy baseline up to ~0.87, since it screens out crops the
detector was unsure about but flagged anyway.
